# 02. Split, images and YOLO dataset

This notebook runs the three steps that follow the data audit (`data-audit.ipynb`, sections 10 and 11):

1. **Split.** `src/split.py` builds our grouped, stratified 70/15/15 split and writes `splits/split_v1.csv`, which goes in the repo.
2. **Images.** `src/fetch_images.py` downloads the images from the signed URLs in the ndjson into the shared Google Drive, resized to a long side of 1280 px by default.
3. **YOLO dataset.** `src/make_yolo_dataset.py` copies the images from Drive to the local disk and writes YOLO labels and `dataset.yaml` for the six core classes.

Where things live: the split CSV is in the repo (`splits/`), the images are in the shared Drive (never in git), and the ndjson stays in `data/` (gitignored, because it holds signed URLs). Steps 2 and 3 can be re-run safely; finished files are skipped.

## Parameters

In [1]:
NDJSON      = "../data/kara-agro-ai-maize.ndjson"
SPLIT_CSV   = "../splits/split_v1.csv"
DRIVE_IMAGES = "G:/My Drive/2026/AI RESEARCH FOUNDATIONS/CAPSTONE/DATASETS/data"   # Colab: "/content/drive/MyDrive/..."
LONG_SIDE   = 1280      # None keeps the originals (about 31 GB)
YOLO_DIR    = "C:/yolo"   # fast local disk, not Drive (Colab: "/content/yolo", rebuilt each session)
NEG_FRACTION = 1.0      # share of 'none healthy' negatives kept in train (val/test keep all)
SEED        = 42

In [2]:
import sys
sys.path.append("..")          # lets the notebook import from src/

# On Colab, mount Drive first (skip when running locally):
# from google.colab import drive; drive.mount("/content/drive")

## 1. Split

Drops the 99 unlabelled shipped-test images, merges "healthy images" into healthy maize, keeps "none healthy" images as background negatives with their boxes dropped, groups images by the loose filename stem, and splits the groups about 70/15/15, stratified on each of the six core classes. The report checks that no group is in two splits, that val and test each have at least 40 egg images, and that the resolution mix is similar per split. It raises an error if a hard check fails.

In [3]:
from src import split
table = split.run(NDJSON, SPLIT_CSV, group_by="stem", seed=SEED)

ndjson: 16543 images, 92932 boxes (audit: 16,543 and 92,932); 99 without boxes dropped
images: 16444   groups: 12578   negatives: 187

[PASS] groups in more than one split: 0

image share per split: {'train': 0.699, 'val': 0.151, 'test': 0.15}  target: {'train': 0.7, 'val': 0.15, 'test': 0.15}

images per class and split:
        egg  larva  frass  damage  healthy  streak  negative
split                                                      
train  210   1015   2357    3594     3223    4342       130
val     45    218    507     769      692     930        29
test    46    218    508     772      692     932        28

boxes per class and split:
        egg  larva  frass  damage  healthy  streak
split                                            
train  715   1057  10519   26749    16196    9620
val    143    231   2402    5575     3550    1971
test   168    223   2296    5752     3279    2016

% of each class's images per split (target 70/15/15):
         egg  larva  frass  damage  healt

**Sensitivity check (optional).** The same split, grouped by capture date instead of filename stem. It is written to a separate file and only reported, never used for training.

In [4]:
table_date = split.run(NDJSON, "../splits/split_v1_dategroups.csv", group_by="date", seed=SEED, strict=False)

ndjson: 16543 images, 92932 boxes (audit: 16,543 and 92,932); 99 without boxes dropped
images: 16444   groups: 1695   negatives: 187

[PASS] groups in more than one split: 0

image share per split: {'train': 0.664, 'val': 0.183, 'test': 0.153}  target: {'train': 0.7, 'val': 0.15, 'test': 0.15}

images per class and split:
        egg  larva  frass  damage  healthy  streak  negative
split                                                      
train  179   1002   2118    3466     3108    4004       124
val     44    230    578     806      735    1359        41
test    78    219    676     863      764     841        22

boxes per class and split:
        egg  larva  frass  damage  healthy  streak
split                                            
train  557   1046   9579   22437    15064    8776
val    110    233   2227    5649     2898    2518
test   359    232   3411    9990     5063    2313

% of each class's images per split (target 70/15/15):
         egg  larva  frass  damage  healt

## 2. Download images to the shared Drive

Downloads only the images listed in the split CSV, resized to `LONG_SIDE`. Files already in the folder are skipped and never overwritten, and each image is written under a hidden temporary name and renamed when complete, so a re-run after an interruption cannot create duplicates or partial files. If many downloads fail with HTTP 403, the signed URLs have expired: re-export the ndjson from Ultralytics, replace the file in `data/`, and run again.

The full download takes hours, so it is easier to run it from a terminal in the repo root than from this cell (it keeps going if the notebook is closed):

```
python -m src.fetch_images --ndjson data/kara-agro-ai-maize.ndjson --out "G:/My Drive/2026/AI RESEARCH FOUNDATIONS/CAPSTONE/DATASETS/data" --split-csv splits/split_v1.csv --workers 6
```

Re-running the cell below is still fine: it only fetches what is missing. Set `limit=50` for a quick trial.

In [5]:
from src import fetch_images
failed = fetch_images.run(NDJSON, DRIVE_IMAGES, split_csv=SPLIT_CSV, long_side=LONG_SIDE, workers=6)
failed.head()

KeyboardInterrupt: 

## 3. Build the YOLO dataset on local disk

Copies the images from Drive to `YOLO_DIR` and writes one label file per image ("none healthy" negatives get an empty file) plus `dataset.yaml`. Training then reads from the local disk, not from Drive. On Colab the local disk is wiped when the session ends, so re-run this cell at the start of each session.

In [ ]:
from src import make_yolo_dataset
make_yolo_dataset.run(SPLIT_CSV, DRIVE_IMAGES, NDJSON, YOLO_DIR, neg_fraction=NEG_FRACTION, seed=SEED)
print(open(f"{YOLO_DIR}/dataset.yaml").read())